In [1]:
from collections import defaultdict
from time import perf_counter
import os

# ------------------------------------------------------------
# OPTIONAL MEMORY MONITORING
# ------------------------------------------------------------

try:
    import psutil

    process = psutil.Process(os.getpid())
    PSUTIL_AVAILABLE = True

except ImportError:
    process = None
    PSUTIL_AVAILABLE = False


# ------------------------------------------------------------
# GLOBAL k-AP CACHE
#
# Key:
#     (bitmask, x)
#
# Value:
#     True  -> adding x creates a k-AP
#     False -> adding x does not create a k-AP
#
# IMPORTANT:
# This cache assumes k and min_value remain unchanged.
# If either changes, clear the cache.
# ------------------------------------------------------------

kap_cache = {}

_NOT_CACHED = object()


# ------------------------------------------------------------
# CREATE DEPTH STATISTICS
# ------------------------------------------------------------

def make_depth_stats():

    return defaultdict(
        lambda: {
            "nodes": 0,

            "cache_hits": 0,
            "cache_misses": 0,

            "choices_tried": 0,

            "ap_prunes": 0,
            "lookahead_prunes": 0,

            "choices_survived": 0,

            "candidates_yielded": 0,
        }
    )


# ------------------------------------------------------------
# BITMASK UTILITY
#
# Number n corresponds to bit n - 1.
#
# Example:
#
#     {1, 4, 6}
#
# becomes:
#
#     bit 0 = 1
#     bit 3 = 1
#     bit 5 = 1
# ------------------------------------------------------------

def value_bit(x: int) -> int:

    return 1 << (x - 1)


# ------------------------------------------------------------
# TEST WHETHER ADDING x CREATES A k-TERM AP
#
# current_mask represents the current partial candidate.
#
# Because values are added in increasing order, x must be
# the largest/final term of any NEW arithmetic progression.
# ------------------------------------------------------------

def creates_kAP(
    current_mask: int,
    x: int,
    k: int,
    cache_stats: dict,
    depth_stats,
    depth: int,
    min_value: int = 1
) -> bool:

    # --------------------------------------------------------
    # CACHE LOOKUP
    # --------------------------------------------------------

    key = (current_mask, x)

    cached_result = kap_cache.get(
        key,
        _NOT_CACHED
    )

    if cached_result is not _NOT_CACHED:

        cache_stats["hits"] += 1
        depth_stats[depth]["cache_hits"] += 1

        return cached_result

    # --------------------------------------------------------
    # CACHE MISS
    # --------------------------------------------------------

    cache_stats["misses"] += 1
    depth_stats[depth]["cache_misses"] += 1

    # Largest possible common difference
    d_max = (x - min_value) // (k - 1)

    # --------------------------------------------------------
    # TEST POSSIBLE COMMON DIFFERENCES
    # --------------------------------------------------------

    for d in range(1, d_max + 1):

        progression_found = True

        # x is the last term, so check:
        #
        #     x-d
        #     x-2d
        #     ...
        #     x-(k-1)d
        #
        # in the bitmask.

        for i in range(1, k):

            previous_value = x - i * d

            bit = 1 << (previous_value - 1)

            if not (current_mask & bit):

                progression_found = False
                break

        if progression_found:

            kap_cache[key] = True

            return True

    # --------------------------------------------------------
    # NO k-AP FOUND
    # --------------------------------------------------------

    kap_cache[key] = False

    return False


# ------------------------------------------------------------
# GENERATE AP-FREE CANDIDATES
#
# Uses:
#
#   1. Immediate AP pruning
#   2. One-step look-ahead pruning
#   3. Bitmask representation
#   4. Cached AP checks
# ------------------------------------------------------------

def generate_AP_free_candidates(
    S,
    size: int,
    k: int,
    stats: dict,
    cache_stats: dict,
    depth_stats
):

    S = list(S)

    if not S:
        return

    min_value = S[0]

    # --------------------------------------------------------
    # RECURSIVE BACKTRACKING SEARCH
    # --------------------------------------------------------

    def backtrack(
        start: int,
        current: list,
        current_mask: int
    ):

        depth = len(current)

        # ----------------------------------------------------
        # NODE VISITED
        # ----------------------------------------------------

        stats["nodes"] += 1
        depth_stats[depth]["nodes"] += 1

        # ----------------------------------------------------
        # SUCCESS
        # ----------------------------------------------------

        if depth == size:

            stats["candidates_yielded"] += 1
            depth_stats[depth]["candidates_yielded"] += 1

            yield tuple(current)

            return

        # ----------------------------------------------------
        # TRY EACH POSSIBLE NEXT VALUE
        # ----------------------------------------------------

        for i in range(start, len(S)):

            x = S[i]

            stats["choices_tried"] += 1
            depth_stats[depth]["choices_tried"] += 1

            # ------------------------------------------------
            # IMMEDIATE AP TEST
            # ------------------------------------------------

            if creates_kAP(
                current_mask,
                x,
                k,
                cache_stats,
                depth_stats,
                depth,
                min_value
            ):

                stats["ap_prunes"] += 1
                depth_stats[depth]["ap_prunes"] += 1

                continue

            # ------------------------------------------------
            # PRETEND x HAS BEEN ADDED
            # ------------------------------------------------

            x_bit = value_bit(x)

            test_mask = current_mask | x_bit

            # Number of additional values needed AFTER x
            needed_after_x = size - (depth + 1)

            # ------------------------------------------------
            # LOOK-AHEAD
            #
            # Count how many later values are individually
            # eligible relative to current + x.
            #
            # IMPORTANT:
            # These values are NOT guaranteed to be mutually
            # compatible.
            #
            # Therefore this is an optimistic upper bound.
            #
            # If even the optimistic count is too small, the
            # branch is impossible.
            # ------------------------------------------------

            eligible_after_x = 0

            if needed_after_x > 0:

                for j in range(i + 1, len(S)):

                    y = S[j]

                    if not creates_kAP(
                        test_mask,
                        y,
                        k,
                        cache_stats,
                        depth_stats,
                        depth,
                        min_value
                    ):

                        eligible_after_x += 1

                        # ------------------------------------
                        # IMPORTANT OPTIMIZATION
                        #
                        # We only need to know whether enough
                        # eligible values exist.
                        #
                        # Once we reach that number, stop
                        # scanning.
                        # ------------------------------------

                        if eligible_after_x >= needed_after_x:
                            break

            # ------------------------------------------------
            # LOOK-AHEAD PRUNE
            # ------------------------------------------------

            if eligible_after_x < needed_after_x:

                stats["lookahead_prunes"] += 1
                depth_stats[depth]["lookahead_prunes"] += 1

                continue

            # ------------------------------------------------
            # x SURVIVES
            # ------------------------------------------------

            stats["choices_survived"] += 1
            depth_stats[depth]["choices_survived"] += 1

            # Keep the list only so we can return/display
            # the actual candidate.

            current.append(x)

            # ------------------------------------------------
            # RECURSE
            #
            # No current_set.add() / remove() is necessary.
            #
            # Integers are immutable, so test_mask is simply
            # passed down to the child.
            # ------------------------------------------------

            yield from backtrack(
                i + 1,
                current,
                test_mask
            )

            # ------------------------------------------------
            # UNDO LIST CHANGE
            # ------------------------------------------------

            current.pop()

    # --------------------------------------------------------
    # START SEARCH
    #
    # Empty set = bitmask 0
    # --------------------------------------------------------

    yield from backtrack(
        start=0,
        current=[],
        current_mask=0
    )


# ------------------------------------------------------------
# CALCULATE r_k(N)
#
# Given:
#
#     previous_max = r_k(N - 1)
#
# only two possibilities exist:
#
#     r_k(N) = previous_max
#
# or:
#
#     r_k(N) = previous_max + 1
#
# Therefore we only search for:
#
#     previous_max + 1
# ------------------------------------------------------------

def r(
    N: int,
    k: int,
    previous_max: int
):

    # --------------------------------------------------------
    # GENERAL SEARCH STATISTICS
    # --------------------------------------------------------

    stats = {

        "nodes": 0,

        "choices_tried": 0,

        "ap_prunes": 0,
        "lookahead_prunes": 0,

        "choices_survived": 0,

        "candidates_yielded": 0,

        "elapsed_seconds": 0.0,
    }

    # --------------------------------------------------------
    # CACHE STATISTICS
    # --------------------------------------------------------

    cache_stats = {

        "hits": 0,
        "misses": 0,
    }

    # --------------------------------------------------------
    # DEPTH STATISTICS
    # --------------------------------------------------------

    depth_stats = make_depth_stats()

    # --------------------------------------------------------
    # TARGET
    # --------------------------------------------------------

    target_size = previous_max + 1

    S = range(
        1,
        N + 1
    )

    start_time = perf_counter()

    # --------------------------------------------------------
    # SEARCH
    # --------------------------------------------------------

    for candidate in generate_AP_free_candidates(
        S,
        target_size,
        k,
        stats,
        cache_stats,
        depth_stats
    ):

        stats["elapsed_seconds"] = (
            perf_counter() - start_time
        )

        return (
            target_size,
            candidate,
            stats,
            cache_stats,
            depth_stats
        )

    # --------------------------------------------------------
    # SEARCH EXHAUSTED
    #
    # No candidate of previous_max + 1 exists.
    # --------------------------------------------------------

    stats["elapsed_seconds"] = (
        perf_counter() - start_time
    )

    return (
        previous_max,
        None,
        stats,
        cache_stats,
        depth_stats
    )


# ------------------------------------------------------------
# PRINT MAIN RESULT
# ------------------------------------------------------------

def print_result(
    N,
    r_value,
    stats
):

    print()

    print(
        f"RESULT FOR N = {N}"
    )

    print(
        "-" * 133
    )

    print(
        f"{'N':>3} | "
        f"{'r_k(N)':>6} | "
        f"{'Seconds':>10} | "
        f"{'Nodes':>12} | "
        f"{'Tried':>12} | "
        f"{'AP Prunes':>12} | "
        f"{'LA Prunes':>12} | "
        f"{'Survived':>12} | "
        f"{'Yielded':>8}"
    )

    print(
        "-" * 133
    )

    print(
        f"{N:>3} | "
        f"{r_value:>6} | "
        f"{stats['elapsed_seconds']:>10.4f} | "
        f"{stats['nodes']:>12,} | "
        f"{stats['choices_tried']:>12,} | "
        f"{stats['ap_prunes']:>12,} | "
        f"{stats['lookahead_prunes']:>12,} | "
        f"{stats['choices_survived']:>12,} | "
        f"{stats['candidates_yielded']:>8,}"
    )


# ------------------------------------------------------------
# PRINT CACHE SUMMARY
# ------------------------------------------------------------

def print_cache_summary(
    cache_stats
):

    hits = cache_stats["hits"]
    misses = cache_stats["misses"]

    total = hits + misses

    if total > 0:

        hit_rate = hits / total

    else:

        hit_rate = 0.0

    print()

    print(
        f"Cache Hits:    {hits:,}"
    )

    print(
        f"Cache Misses:  {misses:,}"
    )

    print(
        f"Cache Hit Rate: {hit_rate:.2%}"
    )

    print(
        f"Cache Entries: {len(kap_cache):,}"
    )

    # --------------------------------------------------------
    # PROCESS MEMORY
    # --------------------------------------------------------

    if PSUTIL_AVAILABLE:

        memory_gb = (
            process.memory_info().rss
            / (1024 ** 3)
        )

        print(
            f"Process Memory: {memory_gb:.3f} GB"
        )

    else:

        print(
            "Process Memory: psutil not installed"
        )


# ------------------------------------------------------------
# PRINT DEPTH STATISTICS
# ------------------------------------------------------------

def print_depth_stats(
    N,
    target_size,
    depth_stats
):

    if not depth_stats:
        return

    # --------------------------------------------------------
    # FIND DEPTH WITH MOST NODES
    # --------------------------------------------------------

    max_node_depth = max(
        depth_stats,
        key=lambda depth:
            depth_stats[depth]["nodes"]
    )

    max_nodes = (
        depth_stats[max_node_depth]["nodes"]
    )

    print()

    print(
        f"{max_node_depth} is max node depth "
        f"({max_nodes:,} nodes)."
    )

    print()

    print(
        f"Depth statistics for N = {N}, "
        f"searching for subset size {target_size}"
    )

    print()

    header = (
        f"{'Depth':>5} | "
        f"{'Nodes':>12} | "
        f"{'Cache Hits':>12} | "
        f"{'Cache Misses':>12} | "
        f"{'Choices Tried':>14} | "
        f"{'AP Prunes':>12} | "
        f"{'LA Prunes':>12} | "
        f"{'Choices Survived':>17} | "
        f"{'Yielded':>8}"
    )

    print(header)

    print(
        "-" * len(header)
    )

    for depth in sorted(depth_stats):

        ds = depth_stats[depth]

        print(
            f"{depth:>5} | "
            f"{ds['nodes']:>12,} | "
            f"{ds['cache_hits']:>12,} | "
            f"{ds['cache_misses']:>12,} | "
            f"{ds['choices_tried']:>14,} | "
            f"{ds['ap_prunes']:>12,} | "
            f"{ds['lookahead_prunes']:>12,} | "
            f"{ds['choices_survived']:>17,} | "
            f"{ds['candidates_yielded']:>8,}"
        )


# ------------------------------------------------------------
# MAIN DRIVER
# ------------------------------------------------------------

if __name__ == "__main__":

    # --------------------------------------------------------
    # CONFIGURATION
    # --------------------------------------------------------

    k = 3

    N_start = 5
    N_end = 75

    # r_3(5) = 4
    #
    # Because we already know this value, begin with it.
    #
    # If you change N_start, make sure max_size contains the
    # EXACT known value r_k(N_start).

    max_size = 4

    # --------------------------------------------------------
    # CACHE POLICY
    #
    # True:
    #     clear cache before each new N
    #
    #     Safer for memory.
    #
    # False:
    #     retain cache across N values
    #
    #     Potentially faster, but memory can grow enormously.
    # --------------------------------------------------------

    CLEAR_CACHE_EACH_N = True

    # --------------------------------------------------------
    # PRINT KNOWN STARTING VALUE
    # --------------------------------------------------------

    print(
        f"N = {N_start}, r_{k}({N_start}) = {max_size}"
    )

    # --------------------------------------------------------
    # COMPUTE N_start + 1 THROUGH N_end
    # --------------------------------------------------------

    for N in range(
        N_start + 1,
        N_end + 1
    ):

        if CLEAR_CACHE_EACH_N:

            kap_cache.clear()

        previous_max = max_size

        (
            max_size,
            candidate,
            stats,
            cache_stats,
            depth_stats
        ) = r(
            N,
            k,
            previous_max
        )

        # ----------------------------------------------------
        # MAIN SUMMARY
        # ----------------------------------------------------

        print_result(
            N,
            max_size,
            stats
        )

        # ----------------------------------------------------
        # CACHE SUMMARY
        # ----------------------------------------------------

        print_cache_summary(
            cache_stats
        )

        # ----------------------------------------------------
        # DEPTH SUMMARY
        # ----------------------------------------------------

        target_size = previous_max + 1

        print_depth_stats(
            N,
            target_size,
            depth_stats
        )

        # ----------------------------------------------------
        # FOUND CANDIDATE
        # ----------------------------------------------------

        if candidate is not None:

            print()

            print(
                f"Candidate: {candidate}"
            )

        print()
        print("=" * 133)

N = 5, r_3(5) = 4

RESULT FOR N = 6
-------------------------------------------------------------------------------------------------------------------------------------
  N | r_k(N) |    Seconds |        Nodes |        Tried |    AP Prunes |    LA Prunes |     Survived |  Yielded
-------------------------------------------------------------------------------------------------------------------------------------
  6 |      4 |     0.0001 |            4 |           19 |            1 |           15 |            3 |        0

Cache Hits:    12
Cache Misses:  40
Cache Hit Rate: 23.08%
Cache Entries: 40
Process Memory: 0.061 GB

1 is max node depth (2 nodes).

Depth statistics for N = 6, searching for subset size 5

Depth |        Nodes |   Cache Hits | Cache Misses |  Choices Tried |    AP Prunes |    LA Prunes |  Choices Survived |  Yielded
--------------------------------------------------------------------------------------------------------------------------------
    0 |            1 

KeyboardInterrupt: 

In [ ]:
if __name__ == "__main__":

    # --------------------------------------------
    # CONFIGURATION
    # --------------------------------------------

    k = 3

    N_start = 5
    N_end = 15

    # Exact known value:
    # r_3(5) = 4
    max_size = 4

    # Clear cache before each new N.
    # This keeps memory growth under control.
    CLEAR_CACHE_EACH_N = True

    # --------------------------------------------
    # PRINT STARTING VALUE
    # --------------------------------------------

    print(
        f"N = {N_start}, "
        f"r_{k}({N_start}) = {max_size}"
    )

    print("=" * 133)

    # --------------------------------------------
    # COMPUTE EACH N
    # --------------------------------------------

    for N in range(N_start + 1, N_end + 1):

        # ----------------------------------------
        # OPTIONAL CACHE RESET
        # ----------------------------------------

        if CLEAR_CACHE_EACH_N:
            kap_cache.clear()

        previous_max = max_size

        # ----------------------------------------
        # RUN SEARCH
        # ----------------------------------------

        (
            max_size,
            candidate,
            stats,
            cache_stats,
            depth_stats
        ) = r(
            N,
            k,
            previous_max
        )

        # ----------------------------------------
        # PRINT MAIN RESULT
        # ----------------------------------------

        print_result(
            N,
            max_size,
            stats
        )

        # ----------------------------------------
        # PRINT CACHE INFORMATION
        # ----------------------------------------

        print_cache_summary(
            cache_stats
        )

        # ----------------------------------------
        # PRINT DEPTH INFORMATION
        # ----------------------------------------

        target_size = previous_max + 1

        print_depth_stats(
            N,
            target_size,
            depth_stats
        )

        # ----------------------------------------
        # PRINT CANDIDATE IF ONE WAS FOUND
        # ----------------------------------------

        if candidate is not None:

            print()

            print(
                f"Candidate: {candidate}"
            )

        print()
        print("=" * 133)